In [1]:
import os

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

In [2]:
os.chdir('../')

In [3]:
import tensorflow as tf

In [4]:
model = tf.keras.models.load_model('artifacts/training/model.keras')

In [5]:
from src.end_to_end_deep_learning_project_yt.entity.config_entity import Config, Params

In [6]:
from pathlib import Path
from pydantic import BaseModel

class EvaluationConfig(BaseModel):
    path_of_model: Path
    training_data: Path
    params_image_size: list
    params_batch_size: int

In [7]:
from src.end_to_end_deep_learning_project_yt.constants import constants
from src.end_to_end_deep_learning_project_yt.utils.common import read_yaml, create_directories, save_json

In [8]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = constants.CONFIG_FILE_PATH,
        params_filepath = constants.PARAMS_FILE_PATH
    ):
        self.config = read_yaml(config_filepath, Config)
        self.params = read_yaml(params_filepath, Params)

        create_directories([self.config.artifacts_root])

    def get_validation_config(self) -> EvaluationConfig:
        eval_config = EvaluationConfig(
            path_of_model='artifacts/training/model.keras',
            training_data='artifacts/data_ingestion/Chicken-fecal-images/',
            params_image_size=self.params.image_size,
            params_batch_size=self.params.batch_size,
        )

        return eval_config

In [9]:
from urllib.parse import urlparse

In [10]:
class Evaluation:
    def __init__(self, config: EvaluationConfig):
        self.config = config

    def _valid_generator(self):
        datagenerator_kwargs = {
            'rescale': 1./255
            , 'validation_split': 0.3
        }

        dataflow_kwargs = {
            'target_size': self.config.params_image_size[:-1]
            , 'batch_size': self.config.params_batch_size
            , 'interpolation': 'bilinear'
        }

        valid_datagenerator = tf.keras.preprocessing.image.ImageDataGenerator(
            **datagenerator_kwargs
        )

        self.valid_generator = valid_datagenerator.flow_from_directory(
            directory=self.config.training_data
            , subset='validation'
            , shuffle=False
            , **dataflow_kwargs
        )

    @staticmethod
    def load_model(path: Path) -> tf.keras.models.Model:
        return tf.keras.models.load_model(path)

    def evaluation(self):
        model = self.load_model(self.config.path_of_model)
        self._valid_generator()
        self.score = model.evaluate(self.valid_generator)

    def save_score(self):
        scores = {'logs': self.score[0], 'accuracy': self.score[1]}
        save_json(path=Path('scores.json'), data=scores)

In [11]:
try:
    config = ConfigurationManager()
    val_config = config.get_validation_config()
    evaluation = Evaluation(val_config)
    evaluation.evaluation()
    evaluation.save_score()
except Exception as e:
    raise

[2026-11-08 16:30:28,590]: INFO - common - yaml file: config/config.yaml loaded successfully
[2026-11-08 16:30:28,591]: INFO - common - yaml file: params.yaml loaded successfully
[2026-11-08 16:30:28,592]: INFO - common - created directory at: artifacts
Found 116 images belonging to 2 classes.
8/8 ━━━━━━━━━━━━━━━━━━━━ 91s 6s/step - accuracy: 0.7586 - loss: 5.4890
[2026-11-08 16:31:59,826]: INFO - common - json file saved at: scores.json
